# B6 · Joins and dissolve: names, mismatches and provinces

> **Time:** about 90 minutes  
> **You need:** Part A, especially A9 · Pandas tables, and Lessons B1 to B5. You should be able to filter a GeoDataFrame, add an area column in km² with `to_crs(32645)`, and draw layers with `ax=ax`.  
> **By the end you can:**
> 1. add the columns of a plain table to a map with an **attribute join** (`.merge`), and check that every row found its partner,
> 2. find and fix a spelling mismatch between two datasets, such as Chitwan and Chitawan,
> 3. merge districts into provinces with **dissolve**, summarise them with `aggfunc`, and draw a first **choropleth** map.

Data rarely arrives in one file. The provinces layer knows each province's shape and code, but not its name. The names live in a small CSV table. The landslide records name their district, but spell one name differently from the boundary layer. In this lesson you join such pieces together, check the joins honestly, and build provinces out of districts.

**Counting lines in this lesson:** the lines that only set up a plot, `fig, ax = plt.subplots()` and `plt.show()`, are not counted in the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 16 of 21 · [Course home](../README.md)  
← Previous: [B5 · Selecting by attribute: finding the districts you need](B05_Selecting_by_attribute.ipynb) · Next: [B7 · Clip and select by location: inside, touching, cut to shape](B07_Clip_and_select_by_location.ipynb) →

## 1. Two tables that belong together

Open the provinces layer and you find seven shapes with a `province_code` such as `NP03`, but no names. The file `province_names.csv` is a plain table with the same codes and their names.

Linking two tables through a column they share is called a **join**. The shared column is the **key**. Here the key is `province_code`.

A join only works when the key values match **exactly** in both tables: same spelling, same capitals, same type. `"NP03"` (text) never matches `3` (a number).

So before any join, look at both keys. `.isin()` from B5 checks whether each value of one key appears in the other.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

districts = gpd.read_file(DATA / "nepal_districts.geojson")
provinces = gpd.read_file(DATA / "nepal_provinces.geojson")
names = pd.read_csv(DATA / "province_names.csv")
slides = pd.read_csv(DATA / "practice_landslides.csv")
print(provinces.columns)
names

In [ ]:
print(provinces["province_code"].isin(names["province_code"]).sum(), "of", len(provinces))

`provinces` has only two columns, `province_code` and `geometry`. `names` has the same seven codes, with a number and a name for each.

All 7 province codes of the layer are found in the names table, so a join will work. `slides` is the landslide CSV, read as a plain pandas table this time. You will join it later in the lesson.

### Practice

**Q1 · one line:** Print the column types of `names` with `.dtypes`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(names.dtypes)
```

`province_code` and `province` are text (`str`), and `province_number` is a whole number (`int64`). The codes are text in both tables, so they can match.
</details>

**Q2 · two lines:** Check that every province code in `names` is used by at least one district. Count the matches with `.isin` and `.sum()`, then print `len(names)` to compare.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(names["province_code"].isin(districts["province_code"]).sum())
print(len(names))
```

Both numbers are 7: every code in the names table appears in the districts layer.
</details>

**Q3 · three lines:** The landslide table `slides` has a `province` column with **names**, not codes. Count the records per province name. Then check how many of the 40 records have a province name that appears in `names["province"]`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(slides["province"].value_counts())
match = slides["province"].isin(names["province"])
print(match.sum(), "of", len(slides))
```

The records are in Bagmati (26), Gandaki (10) and Koshi (4), and all 40 names appear in the names table. A key does not have to be a code: names work too, as long as they are spelled the same.
</details>

## 2. An attribute join with `.merge`

`left.merge(right, on="province_code")` goes through the left table row by row. For each row it finds the row of the right table with the same key, and adds that row's columns.

Joining by matching values in a column is called an **attribute join**, because it uses the table, not the shapes.

**Keep the map on the left.** `provinces.merge(names, ...)` gives a GeoDataFrame that you can draw. `names.merge(provinces, ...)` holds the same values, but comes out as a plain DataFrame.

**Predict:** `provinces` has 2 columns and `names` has 3. They share one. How many columns will the joined table have?

In [ ]:
provinces_named = provinces.merge(names, on="province_code")
print(type(provinces_named))
print(provinces_named.shape)
provinces_named[["province_code", "province_number", "province"]]

In [ ]:
print(type(names.merge(provinces, on="province_code")))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
provinces_named.plot(ax=ax, column="province", legend=True, edgecolor="white")
ax.set_title("The seven provinces of Nepal")
plt.show()

The joined table has 7 rows and 4 columns: the key appears only once, so 2 + 3 − 1 = 4. Each province now has its name and number.

With the plain table on the left, the type is `pandas.DataFrame`. The map is the same data, drawn with the names in the legend. Text columns are listed in alphabetical order, so the legend starts with Bagmati.

### Practice

**Q1 · one line:** Join `names` onto `provinces` again, and print only the `shape` of the result.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(provinces.merge(names, on="province_code").shape)
```

`(7, 4)`: seven provinces, and the three columns of `names` added to `geometry` with the key counted once.
</details>

**Q2 · two lines:** Use `provinces_named` to print the name of the province with code `NP05`. Filter the row, then take the name with `.iloc[0]`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row = provinces_named[provinces_named["province_code"] == "NP05"]
print(row["province"].iloc[0])
```

`NP05` is Lumbini. The filter gives a one-row table; `.iloc[0]` takes the first (and only) value of its `province` column.
</details>

**Q3 · three lines:** Draw the provinces with Karnali highlighted: all provinces in light grey, then Karnali in orange on top. Select Karnali by its **name**, not its code. Add the title "Karnali province".

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
provinces_named.plot(ax=ax, color="lightgrey", edgecolor="white")
provinces_named[provinces_named["province"] == "Karnali"].plot(ax=ax, color="tab:orange")
ax.set_title("Karnali province")
plt.show()
```

After the join you can select by name, which is easier to read than a code. Karnali is the large province in the north-west.
</details>

## 3. Many districts, one province: a many-to-one join

The same names table can be joined onto the **districts**. Now 14 districts share the code `NP01`, and all 14 receive the name Koshi.

This is a **many-to-one** join: many rows on the left match one row on the right. The number of rows on the left must not change: 77 districts in, 77 districts out.

Always compare `len()` before and after a join. If the right table had the same key twice (for example two rows for `NP03`), every Bagmati district would appear twice. The row count would rise without any error message.

In [ ]:
districts_named = districts.merge(names, on="province_code")
print(len(districts), len(districts_named))
districts_named[["district", "province_code", "province"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
districts_named.plot(ax=ax, column="province", legend=True, edgecolor="white", linewidth=0.5)
ax.set_title("Districts coloured by province")
plt.show()

Both counts are 77: no district was lost or doubled. The first rows show the pairs: Achham, Baitadi and Bajhang are in Sudurpashchim, Arghakhanchi in Lumbini, and Baglung in Gandaki.

The map has the same colours as the provinces map in Step 2, but with district borders inside each province.

### Practice

**Q1 · one line:** Count the districts per province **name** in `districts_named`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_named["province"].value_counts()
```

Koshi has 14 districts, Bagmati 13, Lumbini 12, Gandaki 11, Karnali 10, Sudurpashchim 9 and Madhesh 8: the same counts as the codes in B5, now with names.
</details>

**Q2 · two lines:** Which province is Mustang in? Filter `districts_named` for Mustang and print its province name with `.iloc[0]`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mustang = districts_named[districts_named["district"] == "Mustang"]
print(mustang["province"].iloc[0])
```

Mustang is in Gandaki. The join gave every district its province name, so this needs no lookup table.
</details>

**Q3 · three lines:** Add an `area_km2` column to `districts_named` (convert a copy to EPSG:32645, as in B5). Then add up the area per province name with `groupby` and print it from largest to smallest.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_named["area_km2"] = districts_named.to_crs(32645).area / 1000000
by_name = districts_named.groupby("province")["area_km2"].sum()
print(by_name.sort_values(ascending=False).round(1))
```

Karnali is the largest province in this data, about 30,855.8 km², and Madhesh the smallest, about 9,587.5 km². `groupby` adds up the numbers, but it does not merge the shapes. Step 6 does that.
</details>

## 4. Checking a join: `how=` and `indicator=True`

By default, `.merge` makes an **inner join**: it keeps only the rows whose key is found in **both** tables. Rows without a partner vanish, and no message tells you.

`how="left"` makes a **left join**: every row of the left table is kept. Where no partner was found, the new columns are filled with `NaN`, pandas' mark for a **missing value**. `.isna()` gives `True` for each missing value.

![Venn diagrams of the four join types: left, right, inner and outer](../images/join_types.jpg)

*Join types. A left join keeps everything from the left table; an inner join keeps only the overlap. Image: the pyGIS project by Michael Mann (GPL-3.0).*

`indicator=True` adds a column `_merge` that says, for each row, where it was found: `both`, `left_only` or `right_only`. It is the quickest honest check of any join.

Below, the landslide records are counted per district and the counts are checked against the district layer. `.reset_index()` turns the district names of `value_counts()` back into a normal column, so they can be used as a key.

In [ ]:
counts = slides["district"].value_counts().reset_index()
counts

In [ ]:
check = counts.merge(districts[["district", "district_code"]], on="district", how="left", indicator=True)
check

In [ ]:
inner = slides.merge(districts[["district", "district_code"]], on="district")
print(len(slides), len(inner))

`counts` has 8 rows, with the columns `district` and `count`. In `check`, seven districts are `both`, but **Chitwan** is `left_only` and its `district_code` is `NaN`: the district layer has no district called Chitwan.

The inner join is worse. It quietly keeps 31 of the 40 landslide records and drops all 9 Chitwan records. A count, a total area or a map made from it would be wrong, and nothing would warn you. Always compare row counts and look at `_merge`.

### Practice

**Q1 · one line:** Count the values in the `_merge` column of `check`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
check["_merge"].value_counts()
```

7 rows are `both` and 1 is `left_only`. The `right_only` count is 0, because a left join never adds rows that exist only in the right table.
</details>

**Q2 · two lines:** Show only the rows of `check` where `_merge` is **not** `both`, with the columns `district` and `_merge`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
problems = check[check["_merge"] != "both"]
problems[["district", "_merge"]]
```

Only Chitwan is left. Filtering for everything that is not `both` is a good habit: on a large table it shows you exactly which keys to fix.
</details>

**Q3 · three lines:** Now go the other way: left-join `counts` onto **all 77 districts** (`districts[["district", "geometry"]]` on the left). Print how many rows the result has and how many districts have a missing `count`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
all_districts = districts[["district", "geometry"]].merge(counts, on="district", how="left")
print(len(all_districts))
print(all_districts["count"].isna().sum())
```

The result keeps all 77 districts, and 70 have no count. Only 7 districts found a count, because Chitawan did not match Chitwan. The other 69 have no landslide records at all.
</details>

## 5. Fixing a mismatch: Chitwan and Chitawan

The landslide file writes **Chitwan**, the common spelling. The boundary layer writes **Chitawan**, following its source. Neither is wrong, but a join needs one spelling.

First find the spelling the map uses. `.str.contains("Chit")` from B5 is a good way.

Then change the **small table** to match the **reference layer**, not the other way round. Other files you join later will use the layer's spelling too.

`.replace({"old": "new"})` swaps values in a column. The curly brackets hold a dictionary (A4): each key is replaced by its value. Values that are not in the dictionary stay as they are. The file on disk is not changed; only the table in memory is.

In [ ]:
print(districts.loc[districts["district"].str.contains("Chit"), "district"])

In [ ]:
slides["district"] = slides["district"].replace({"Chitwan": "Chitawan"})
print(slides["district"].value_counts())

In [ ]:
fixed = slides.merge(districts[["district", "district_code"]], on="district", how="left", indicator=True)
print(fixed["_merge"].value_counts())

The layer spells it **Chitawan**. After `.replace`, the landslide table has 9 records in Chitawan and none in Chitwan.

The check now finds all 40 records in `both`. Write the fix in code, as here, rather than editing the CSV by hand. Anyone who reruns the notebook gets the same correction, and can see that it was made.

### Practice

**Q1 · one line:** Count the records in `slides` whose district is now `Chitawan`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((slides["district"] == "Chitawan").sum())
```

9 records: the same 9 that were called Chitwan before. `.replace` changed the spelling, not the number of records.
</details>

**Q2 · two lines:** Repeat the inner join of Step 4 with the fixed `slides`. Store it in `inner_fixed` and print its length.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
inner_fixed = slides.merge(districts[["district", "district_code"]], on="district")
print(len(inner_fixed))
```

All 40 records survive the inner join now, compared with 31 before the fix.
</details>

**Q3 · three lines:** Count the landslides per district again from the fixed `slides`, left-join the counts onto all districts, and replace missing counts with 0 using `.fillna(0)`. Print how many districts have at least one landslide.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
counts_fixed = slides["district"].value_counts().reset_index()
per_district = districts.merge(counts_fixed, on="district", how="left")
per_district["count"] = per_district["count"].fillna(0)
print((per_district["count"] > 0).sum())
```

8 districts have records, matching the 8 names in the landslide file. `.fillna(0)` is right here because a district with no records really has zero practice landslides, not an unknown number.
</details>

## 6. Dissolve: merging districts into provinces

In A9 you used `groupby` to add up numbers per group. **Dissolve** does the same for shapes: it merges the shapes of all rows that share a value into one shape, and removes the borders inside.

`districts.dissolve(by="province_code")` gives one row per province code. The province codes become the row labels (the index), as with `groupby`.

`aggfunc` says how to summarise the other columns. `aggfunc={"district": "count", "area_km2": "sum"}` counts the districts and adds up their areas. Without `aggfunc`, each province keeps the values of its first district.

Compute areas in UTM 45N first, as always. `.rename(columns={"district": "n_districts"})` then gives the counted column a name that says what it holds, and `.reset_index()` turns the codes back into a column.

In [ ]:
districts_utm = districts.to_crs(32645)
districts_utm["area_km2"] = districts_utm.area / 1000000

by_province = districts_utm.dissolve(by="province_code", aggfunc={"district": "count", "area_km2": "sum"})
by_province = by_province.rename(columns={"district": "n_districts"}).reset_index()
by_province[["province_code", "n_districts", "area_km2"]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
by_province.plot(ax=ax, color="lightgrey", edgecolor="black")
ax.set_title("Provinces made by dissolving districts (UTM 45N)")
plt.show()

Seven rows, each with its number of districts and its total area. The counts, from 14 in `NP01` to 8 in `NP02`, match B5.

The map shows the seven provinces with no district borders inside. The axes are now in metres, because the shapes are in UTM 45N.

The far west even has negative eastings. Nepal spans UTM zones 44N and 45N, split at 84°E, and the west lies outside zone 45N. As in B4, one zone for the whole country stretches the west slightly; for precise national areas an equal-area projection is better.

### Practice

**Q1 · one line:** Dissolve **all** districts into one shape by calling `.dissolve()` with no `by`, and draw it.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_utm.dissolve().plot(color="lightgrey", edgecolor="black", figsize=(9, 5))
```

With no `by`, every row belongs to one group, so you get a single outline of Nepal with no inner borders.
</details>

**Q2 · two lines:** Dissolve all districts into one row called `nepal`, then print its area in km², rounded to a whole number. Compare it with the sum of the district areas.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
nepal = districts_utm.dissolve()
print(round(nepal.area.iloc[0] / 1000000), round(districts_utm["area_km2"].sum()))
```

Both are 148,017 km². The districts fit together with no gaps or overlaps, so the area of the whole equals the sum of the parts.
</details>

**Q3 · three lines:** Make a column `west` that is `True` for districts in `NP05`, `NP06` or `NP07` (use `.isin`). Dissolve by it, adding up `area_km2`, and print the two areas.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_utm["west"] = districts_utm["province_code"].isin(["NP05", "NP06", "NP07"])
halves = districts_utm.dissolve(by="west", aggfunc={"area_km2": "sum"})
print(halves["area_km2"].round())
```

The western three provinces cover about 70,088 km² and the other four about 77,929 km². You can dissolve by any column, including one you have made yourself.
</details>

## 7. Checking your dissolve against the provinces file

The file `nepal_provinces.geojson` was made by dissolving the same districts. So your `by_province` should match it exactly. Checking a result against a trusted copy is a good habit, whenever you have one.

Three checks, from simple to strict:

1. the same number of rows and the same codes,
2. the same areas, after converting both to UTM 45N,
3. the same shapes: `a.geom_equals(b)` gives `True` for each pair of shapes that are identical, row by row.

Make sure both tables are in the same row order before comparing them row by row. Here both are sorted by province code.

In [ ]:
provinces_utm = provinces.to_crs(32645)
provinces_utm["file_km2"] = provinces_utm.area / 1000000

compare = by_province[["province_code", "area_km2"]].merge(
    provinces_utm[["province_code", "file_km2"]], on="province_code")
compare["difference_km2"] = (compare["area_km2"] - compare["file_km2"]).abs().round(3)
compare

In [ ]:
print(by_province.geom_equals(provinces_utm.geometry))

In [ ]:
provinces_final = by_province.merge(names, on="province_code")
provinces_final[["province_code", "province", "n_districts", "area_km2"]]

`.abs()` turns negative numbers positive, so `difference_km2` is the size of each difference. All seven are 0.0 km² after rounding to 0.001 km², and `geom_equals` is `True` for every province. Your dissolve and the file agree exactly.

If the file had been made from older boundaries, you would see differences, and you would need to decide which version to trust.

The last cell joins the names onto your dissolved provinces. `by_province` is on the left, so `provinces_final` is still a GeoDataFrame, with shapes, names, district counts and areas together.

### Practice

**Q1 · one line:** Check that no district was lost in the dissolve: print the sum of the `n_districts` column of `by_province`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(by_province["n_districts"].sum())
```

It prints 77. Every district was counted in exactly one province.
</details>

**Q2 · two lines:** Sort `provinces_final` by area from largest into `ranked`, then show its `province`, `n_districts` and `area_km2` columns.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
ranked = provinces_final.sort_values("area_km2", ascending=False)
ranked[["province", "n_districts", "area_km2"]]
```

Karnali comes first with about 30,855.8 km² from only 10 districts, and Madhesh last with about 9,587.5 km². Koshi has the most districts but is only the second largest.
</details>

**Q3 · three lines:** Save `provinces_final` to `provinces_final.gpkg` in `OUT` (GeoPackage, as in B3). Read it back and print its number of rows and its CRS.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces_final.to_file(OUT / "provinces_final.gpkg")
back = gpd.read_file(OUT / "provinces_final.gpkg")
print(len(back), back.crs)
```

The file holds 7 provinces in `EPSG:32645`: the joined names and the summary columns are saved with the shapes.
</details>

## 8. A first choropleth: provinces by area

A **choropleth** map colours each area by a number, such as its size or its number of landslides. With GeoPandas, `column="area_km2"` picks the number.

`cmap` chooses the colour map. For amounts that go from small to large, use a **sequential** colour map such as `"Greens"` or `"viridis"`: light means little and dark means a lot. Avoid rainbow colours, as in A10.

`legend=True` adds a colour bar, and `legend_kwds={"label": "Area (km²)"}` gives it a label with its unit.

The UTM axis numbers mean little to most readers, so `ax.set_axis_off()` hides them. B11 adds a scale bar and a north arrow instead.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
provinces_final.plot(ax=ax, column="area_km2", cmap="Greens", edgecolor="black",
                     legend=True, legend_kwds={"label": "Area (km²)"})
ax.set_title("Area of Nepal's provinces (from simplified boundaries)")
ax.set_axis_off()
plt.show()

Karnali, in the north-west, is the darkest: it is the largest province. Madhesh, the narrow strip in the south-east, is the lightest.

A choropleth of **totals**, like area, always makes large provinces stand out. In B10 you will map densities, such as landslides per 100 km². You will also see how the choice of colour classes changes a map's story.

### Practice

**Q1 · one line:** Draw a choropleth of `provinces_final` coloured by `n_districts`, with a legend, using the colour map `"viridis"`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces_final.plot(column="n_districts", cmap="viridis", legend=True, edgecolor="white", figsize=(9, 5))
```

Koshi (14 districts) is brightest and Madhesh (8) darkest. `figsize=` inside `.plot()` sets the size when you do not make the axes yourself. In `"viridis"` the colours run from dark purple for small values to yellow for large ones.
</details>

**Q2 · two lines:** Draw a choropleth of the **districts** in `districts_utm` coloured by `area_km2`, using `"Greens"`, with a labelled legend and no axes.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
districts_utm.plot(ax=ax, column="area_km2", cmap="Greens", edgecolor="white", legend=True, legend_kwds={"label": "Area (km²)"})
ax.set_axis_off()
plt.show()
```

Dolpa and Humla in the north-west stand out as the darkest; the small Kathmandu Valley districts are almost white.
</details>

**Q3 · three lines:** Draw the provinces coloured by area again, then draw the province **outlines** of `provinces_final` on top in black with `color="none"`, and add a title.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
provinces_final.plot(ax=ax, column="area_km2", cmap="Greens", legend=True)
provinces_final.plot(ax=ax, color="none", edgecolor="black", linewidth=1.5)
ax.set_title("Province areas with outlines")
plt.show()
```

`color="none"` draws no fill, only the edges. Drawing outlines as a separate top layer keeps the borders crisp, whatever the fill colours below.
</details>

## 9. When it breaks: a key that is not there

This cell tries to join the names table onto the provinces using the column `province`. Run it.

In [ ]:
provinces.merge(names, on="province")

The message is long. Scroll to the last line: `KeyError: 'province'`.

As in A9, a `KeyError` means "no column with this name". `names` has a `province` column, but `provinces` does not; it has only `province_code`. With `on=`, the key column must exist, with the same name, in **both** tables.

Two fixes:

- join on a column both tables share: `on="province_code"`,
- or, when the key has different names in the two tables, name each side: `left_on="code_in_left", right_on="code_in_right"`.

**Try it:** fix the cell so it gives 7 rows.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Landslides per province

Count the landslide records per province **name** in `slides`, with `.reset_index()`. Left-join the counts onto `provinces_final` using the key `province`, and replace missing counts with 0. Print the province names with their counts, then draw a choropleth of the counts with a labelled legend.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
per_province = slides["province"].value_counts().reset_index()
prov_slides = provinces_final.merge(per_province, on="province", how="left")
prov_slides["count"] = prov_slides["count"].fillna(0)
print(prov_slides[["province", "count"]])

fig, ax = plt.subplots(figsize=(9, 5))
prov_slides.plot(ax=ax, column="count", cmap="Oranges", edgecolor="black",
                 legend=True, legend_kwds={"label": "Practice landslide records"})
ax.set_axis_off()
plt.show()
```

Bagmati has 26 records, Gandaki 10, Koshi 4, and the other four provinces 0. A left join keeps all seven provinces; an inner join would have dropped the four with no records, and they would vanish from the map. Remember these are made-up practice records.
</details>

### Review 2 · Landslides per district, after the fix

Use the fixed `slides` (Chitawan). Count records per district, left-join them onto `districts`, fill missing counts with 0, and print the 8 districts with the most records. Then map the counts over all districts.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
counts_fixed = slides["district"].value_counts().reset_index()
per_district = districts.merge(counts_fixed, on="district", how="left")
per_district["count"] = per_district["count"].fillna(0)
print(per_district.sort_values("count", ascending=False)[["district", "count"]].head(8))

fig, ax = plt.subplots(figsize=(9, 5))
per_district.plot(ax=ax, column="count", cmap="Oranges", edgecolor="lightgrey", legend=True)
ax.set_title("Practice landslide records per district")
plt.show()
```

Chitawan and Dolakha have 9 records each, then Rasuwa and Myagdi 5, Gorkha and Taplejung 4, Sindhupalchok 3 and Kaski 1. Before the fix, Chitawan would have shown 0 on this map: a silent join error can make a whole district disappear from a result.
</details>

### Review 3 · Provinces by name, in one dissolve

Dissolve `districts_named` (from Step 3) by `province`, counting the districts with `aggfunc={"district": "count"}`. Rename the column to `n_districts`, sort by it from largest, and print it. Then draw the result with each province in its own colour and a legend.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
by_name_shapes = districts_named.dissolve(by="province", aggfunc={"district": "count"})
by_name_shapes = by_name_shapes.rename(columns={"district": "n_districts"}).reset_index()
print(by_name_shapes.sort_values("n_districts", ascending=False)[["province", "n_districts"]])

fig, ax = plt.subplots(figsize=(9, 5))
by_name_shapes.plot(ax=ax, column="province", legend=True, edgecolor="white")
ax.set_title("Provinces dissolved from named districts")
plt.show()
```

Koshi has 14 districts and Madhesh 8, as before. Once the names are joined, you can dissolve by the name directly, so you never need the codes in the result. This dissolve stays in degrees, which is fine: it only merges shapes and counts rows, it measures nothing.
</details>

### Review 4 · A complete join, checked

Join `names` onto `districts` with `how="left"` and `indicator=True`. Print the number of rows, the counts of `_merge`, and the number of missing province names. Finish with one sentence (as a comment) saying whether the join is safe to use.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
joined = districts.merge(names, on="province_code", how="left", indicator=True)
print(len(joined))
print(joined["_merge"].value_counts())
print(joined["province"].isna().sum())
# 77 rows, all "both", no missing names: the join is complete and safe to use.
```

77 rows, all `both`, and 0 missing names. Three numbers, printed in three lines, are enough to trust a join. Make this check a habit for every join you write.
</details>

## Recap

- An **attribute join** links two tables through a shared **key** column: `left.merge(right, on="key")`. Keep the GeoDataFrame on the left so the result is still a map.
- The default **inner join** silently drops rows with no partner. Use `how="left"` to keep every row, and `indicator=True` to see `both`, `left_only` and `right_only`.
- Check every join: compare `len()` before and after, count `_merge`, and count missing values with `.isna().sum()`.
- Fix spelling mismatches in code with `.replace({"Chitwan": "Chitawan"})`, changing the small table to match the reference layer. `.fillna(0)` fills counts that are truly zero.
- `dissolve(by=..., aggfunc={...})` merges shapes per group, like `groupby` for geometry. A **choropleth** colours areas by a number; use a sequential colour map and a labelled legend.

### Words you learned

| Word | Meaning |
| --- | --- |
| join | Linking two tables through a column they share |
| key | The shared column used to match rows in a join |
| attribute join | A join that matches rows by values in a column, not by location |
| many-to-one | A join where many left rows match the same right row, like districts to provinces |
| inner join | Keeps only rows whose key is found in both tables (the default) |
| left join | Keeps every row of the left table; unmatched rows get missing values |
| `NaN` | pandas' mark for a missing value |
| `indicator=True` | Adds a `_merge` column saying where each row was found |
| `.reset_index()` | Turns the row labels back into a normal column |
| `.replace({old: new})` | Swaps values in a column |
| `.fillna(0)` | Replaces missing values with 0 |
| dissolve | Merges the shapes of rows that share a value into one shape |
| `aggfunc` | How dissolve summarises the other columns, such as `"count"` or `"sum"` |
| `geom_equals` | `True` where two shapes are exactly the same |
| choropleth | A map that colours each area by a number |
| sequential colour map | Colours that run from light (little) to dark (a lot) |

## Next

**B7 · Clip and select by location:** choose features by where they are, not by their attributes: the districts that touch Kathmandu, the landslides inside one district, and everything inside a box.

<!-- pygile-nav -->
**PyGILE Course** · lesson 16 of 21 · [Course home](../README.md)  
← Previous: [B5 · Selecting by attribute: finding the districts you need](B05_Selecting_by_attribute.ipynb) · Next: [B7 · Clip and select by location: inside, touching, cut to shape](B07_Clip_and_select_by_location.ipynb) →